In [ ]:
#########################################
##                                     ##
##               IMPORTS               ##
##                                     ##
#########################################

!pip install transformers
!pip install contractions
!pip install pyspellchecker
!pip install optuna

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import DistilBertTokenizer, DistilBertModel, DistilBertForSequenceClassification
# OPTIONAL: if you want to have more information on what's happening, activate the logger as follows
# import logging
# logging.basicConfig(level=logging.INFO)

In [ ]:
#############################################
##                                         ##
##              DATA FETCHING              ##
##                                         ##
#############################################

data_dir = "./data/"
# data_dir = "/kaggle/input/ai-2-dl-for-nlp-2025-homework-3-distil-bert/"
train_df = pd.read_csv(f"{data_dir}train_dataset.csv")
# train_df = pd.read_csv(f"{data_dir}train_dataset.csv").sample(n=1000, random_state=42)
val_df = pd.read_csv(f"{data_dir}val_dataset.csv")
# val_df = pd.read_csv(f"{data_dir}val_dataset.csv").sample(n=1000, random_state=42)
test_df = pd.read_csv(f"{data_dir}test_dataset.csv")

# debugging
train_df.head()

In [ ]:
#############################################
##                                         ##
##           DATA PRE-PROCESSING           ##
##                                         ##
#############################################

import re
import html
import contractions
from spellchecker import SpellChecker

emoji_dict = {
    r":\)": "smile",
    r":D": "smile",
    r";\)": "wink",
    r":\(": "sad",
    r":'\(": "cry",
    r":P": "playful",
    r"XD": "laugh",
    r"xD": "laugh",
    r"<3": "love",
    r":O": "surprise"
}

def handle_simple_emojis(text):
    for pattern, replacement in emoji_dict.items():
        text = re.sub(pattern, replacement, text)
    return text

def remove_contractions(text):
    # expand contractions (e.g haven't -> have not)
    return contractions.fix(text)

# initialize spell checker
spell = SpellChecker(distance=1)

def my_spellcheck(text):
    words = text.split()
    corrected_words = []
    for word in words:
        # if it's wrong
        if word not in spell:
            test_word = spell.correction(word)
            # found correction
            if test_word is not None:
                corrected_words.append(test_word)
            # didn't find correction
            else:
                corrected_words.append(word)
        # else, if it's correct, append it to the corrected
        else:
            corrected_words.append(word)
    return " ".join(corrected_words)

def text_cleanup(text):
    text = text.lower() # lowercase everything
    text = handle_simple_emojis(text) # handling basic emojis
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', 'email', text) # handle email adresses
    text = re.sub(r'http[s]?://\S+', 'url', text) # handle urls
    text = re.sub(r'@\w+', 'username', text) # handle mentions
    text = re.sub(r'#(\w+)', r'\1', text) # handle hashtags: remove the symbol, keep the text
    text = html.unescape(text) # handling html tags
    text = remove_contractions(text) # handling contractions (haven't -> have not)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text) # remove anything that is not english, a number or a whitespace
    # text = my_spellcheck(text) # spellchecking using pyspellchecker
    return text

# apply cleanup to all datasets
train_df["Text"] = train_df["Text"].apply(text_cleanup)
val_df["Text"] = val_df["Text"].apply(text_cleanup)
test_df["Text"] = test_df["Text"].apply(text_cleanup)

train_df.head()

In [ ]:
##############################################
##                                          ##
##             MODEL DEFINITION             ##
##                                          ##
##############################################

import random
import numpy as np

# Load pre-trained model tokenizer (vocabulary)
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Load pre-trained model (weights)
seed_val = 42

random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# Put the model in "evaluation" mode, meaning feed-forward operation.
model.eval()

In [ ]:
from torch.utils.data import Dataset

class TweetDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        text = self.df.iloc[index]['Text']
        label = self.df.iloc[index]['Label']
        
        encoding = tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


In [ ]:
from torch.utils.data import DataLoader

# dataset for train and val
train_dataset = TweetDataset(train_df, tokenizer, max_len=128)
val_dataset = TweetDataset(val_df, tokenizer, max_len=128)

# Dataloader for batches
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# epochs = 15
epochs = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CUDA available:", torch.cuda.is_available())
print("Device in use:", device)
print("CUDA device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
##########################################
###                                    ###
###               OPTUNA               ###
###                                    ###
##########################################

import optuna
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-6, 5e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)

    seed_val = 42
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)

    model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2).to(device)

    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay, eps = 1e-8)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer,
                                                num_warmup_steps=0,
                                                num_training_steps=total_steps)

    # for early stopping
    best_val_loss = float('inf')
    epochs_no_improve = 0
    patience = 2
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        print(f"\n========== Epoch {epoch+1}/{epochs} ==========")
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)

            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            train_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = correct / total

        # Validation
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(input_ids=input_ids,
                                attention_mask=attention_mask,
                                labels=labels)

                val_loss += outputs.loss.item()
                preds = torch.argmax(outputs.logits, dim=1)
                correct_val += (preds == labels).sum().item()
                total_val += labels.size(0)

        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = correct_val / total_val
        # early stopping logic
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

        trial.set_user_attr("best_epoch", epoch + 1 - epochs_no_improve)
        
        print(f"Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f} - Accuracy: {train_accuracy:.4f}")
        print(f"Validation - Loss: {avg_val_loss:.4f} - Accuracy: {val_accuracy:.4f}")
        
    return best_val_loss


# study = optuna.create_study(direction="minimize", study_name = 'lr&wd_minim_study')
# study.optimize(objective, n_trials=10)

# print("Best hyperparameters:", study.best_params)

In [ ]:
##############################################
##                                          ##
##          (FINAL) MODEL TRAINING          ##
##                                          ##
##############################################

from sklearn.metrics import accuracy_score, classification_report

# best hyperparameters given by the output of the optuna study
# best_lr = study.best_params["learning_rate"]
# best_wd = study.best_params["weight_decay"]
# best_trial = study.best_trial
# best_epochs = best_trial.user_attrs["best_epoch"]
best_epochs = epochs

best_lr = 1.293040592451508e-05
best_wd = 2.1436761573115605e-05

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2).to(device)
optimizer = AdamW(model.parameters(), lr=best_lr, weight_decay=best_wd, eps = 1e-8)
total_steps = len(train_loader) * epochs
# Create the learning rate scheduler to dynamically adjust the learning rate.
scheduler = get_linear_schedule_with_warmup(optimizer,
                                        num_warmup_steps = 0,
                                        num_training_steps = total_steps)

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    correct = 0
    total = 0
    print(f"========== Epoch {epoch+1}/{epochs} ==========")
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)

        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()

        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = epoch_loss / len(train_loader)
    train_accuracy = correct / total
    train_losses.append(avg_loss)
    train_accuracies.append(train_accuracy)

    # validation
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)

            loss = outputs.loss
            val_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct_val / total_val
    val_losses.append(avg_val_loss)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} - Accuracy: {train_accuracy:.4f}")
    print(f"Validation - Loss: {avg_val_loss:.4f} - Accuracy: {val_accuracy:.4f}")

print("=========== Training complete! ===========")


In [ ]:
################################################
##                                            ##
##           VALIDATION & EVALUATION          ##
##                                            ##
################################################

# evaluation on validation dataset
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask)
        
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


accuracy = accuracy_score(all_labels, all_preds)
print(f"\nAccuracy: {accuracy:.4f}")
print("Classification Report:\n", classification_report(all_labels, all_preds, digits=4))

In [ ]:
#######################################
####                               ####
####             PLOTS             ####
####                               ####
#######################################

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# learning curves

epochs_range = range(1, len(train_losses) + 1)

plt.figure(figsize=(10, 4))

# Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, 'bo-', label='Training Loss')
plt.plot(epochs_range, val_losses, 'ro-', label='Validation Loss')
plt.axvline(x=best_epochs, color='green', linestyle='--', label='Best Epoch')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_accuracies, 'bo-', label='Training Accuracy')
plt.plot(epochs_range, val_accuracies, 'ro-', label='Validation Accuracy')
plt.axvline(x=best_epochs, color='green', linestyle='--', label='Best Epoch')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("learning_curves.png", dpi=120)
plt.show()

# confusion matrix

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="OrRd", xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.savefig("confusion_matrix.png")
plt.show()

In [ ]:
##############################################
##                                          ##
##             KAGGLE SUBMISSION            ##
##                                          ##
##############################################

class TestDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        text = self.df.iloc[index]['Text']
        encoding = tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }


test_dataset = TestDataset(test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=32)

# making predictions
model.eval()
predictions = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask)

        preds = torch.argmax(outputs.logits, dim=1)
        predictions.extend(preds.cpu().numpy())

submission_df = pd.DataFrame({
    "ID": test_df["ID"],
    "Label": predictions
})

submission_df.to_csv("submission.csv", index=False)
print("Created submission.csv")